# DATA VORTEX — Round 1 Phase 1
## Cleaning + Validation Pipeline

This notebook runs the full, reproducible pipeline end to end:
1. Clean `data/raw/Social_Engine_Users.csv` → `data/cleaned/Social_Engine_Users_Cleaned.csv`
2. Clean `data/raw/Social_Engine_Posts_Corrupted.csv` → `data/cleaned/Social_Engine_Posts_Cleaned.csv`
3. Rebuild `audit/cleaning_summary.json` from the real cleaned output
4. Validate the final artifacts

Every number in `reports/EDA_Report.md` and `audit/cleaning_summary.json` is produced by this pipeline — nothing is hand-typed.

In [ ]:
import sys
sys.path.append('../src')

from clean_users import clean_users
from clean_posts import clean_posts
import pandas as pd

### Step 1: Clean the users dataset

In [ ]:
users_df, users_audit = clean_users()
users_df.to_csv('../data/cleaned/Social_Engine_Users_Cleaned.csv', index=False)
users_audit

### Step 2: Clean the posts dataset

See `src/clean_posts.py` for the full documented rule set (structural validity, deduplication, per-platform median imputation for likes, IQR outlier flagging).

In [ ]:
posts_df, posts_audit = clean_posts()
posts_df.to_csv('../data/cleaned/Social_Engine_Posts_Cleaned.csv', index=False)
posts_audit

### Step 3: Rebuild the audit JSON (also runnable directly as `python src/build_audit.py`)

In [ ]:
import subprocess
result = subprocess.run(['python3', '../src/build_audit.py'], capture_output=True, text=True)
print(result.stdout[-500:])
print(result.stderr[-500:])

### Step 4: Validate final artifacts

In [ ]:
result = subprocess.run(['python3', '../src/validate_submission.py'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

### Quick EDA sanity check

In [ ]:
posts_df['timestamp'] = pd.to_datetime(posts_df['timestamp'])
posts_df.groupby('platform')['total_engagement'].agg(['count', 'mean', 'median']).round(2)

In [ ]:
markers = ['<div>', '<br>', '&amp;', 'Ã©']
{m: int(posts_df.text_content.str.contains(m, regex=False, na=False).sum()) for m in markers}